In [ ]:
!pip install faiss-gpu-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 40.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
import os
import cv2
import json
import glob
from tqdm.notebook import tqdm

In [ ]:
!rsync -a --info=progress2 "/content/drive/MyDrive/Datasets/DINO_DatasetA.zip" "/content"
import zipfile
with zipfile.ZipFile("DINO_DatasetA.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset-A/")

!rsync -a --info=progress2 "/content/drive/MyDrive/Datasets/DINO_DatasetB.zip" "/content"
import zipfile
with zipfile.ZipFile("DINO_DatasetB.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset-B/")

!rsync -a --info=progress2 "/content/drive/MyDrive/Datasets/DINO_DatasetC.zip" "/content"
import zipfile
with zipfile.ZipFile("DINO_DatasetC.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset-C/")

     29,425,120 100%   52.49MB/s    0:00:00 (xfr#1, to-chk=0/1)
     67,949,686 100%   75.14MB/s    0:00:00 (xfr#1, to-chk=0/1)
    101,143,907 100%   59.30MB/s    0:00:01 (xfr#1, to-chk=0/1)


In [ ]:
!rsync -a --info=progress2 "/content/drive/MyDrive/FAISS_index/DINOv2/Dataset_B/vector.index.paths.json" "./"
!rsync -a --info=progress2 "/content/drive/MyDrive/FAISS_index/DINOv2/Dataset_B/vector.index" "./"

        185,667 100%  145.82MB/s    0:00:00 (xfr#1, to-chk=0/1)
      3,758,186 100%   45.55MB/s    0:00:00 (xfr#1, to-chk=0/1)


In [ ]:
from transformers import AutoModelForImageClassification, AutoImageProcessor

#our ViT-S DINOv2, ImageNet weights are default
dinov2_vits14 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14",pretrained=True)

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


In [ ]:
from typing import Optional, List
#assumes we're passed a train, val, or test set, and that all images are within class folders
def generate_dict_from_set(embeddings: Optional[List] = None, path_to_dataset = None):
  index = 0 if embeddings is None else int(embeddings[-1]['id']) + 1
  all_embeddings = [] if embeddings is None else embeddings
  for folder in os.listdir(path_to_dataset):
    label = folder
    fold_path = os.path.join(path_to_dataset, folder)
    for file in os.listdir(fold_path):
      filepath = os.path.join(fold_path, file)
      all_embeddings.append({'id': index, 'label':label, 'path': filepath})
      index += 1
  return all_embeddings

In [ ]:
def get_label_path_index(dataset_entry, index):
  id = dataset_entry['id']
  id = np.array([id], dtype='int64') #needs to be a 1d nparray for faiss
  label = dataset_entry['label']
  path = dataset_entry['path']
  return id, label, path

In [ ]:
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small", use_fast=True)

def load_image(img: str) -> torch.Tensor:
    """
    Load an image and return a tensor that can be used as an input to DINOv2.
    """
    img = Image.open(img)

    inputs = processor(images=img, return_tensors="pt")

    return inputs["pixel_values"]

In [ ]:
import faiss
import json

#loading from our saved index
#https://towardsdatascience.com/building-an-image-similarity-search-engine-with-faiss-and-clip-2211126d08fa/ again
def load_faiss_index(index_path):
    index = faiss.read_index(index_path)
    with open(index_path + '.paths.json', 'r') as f:
        image_paths = json.load(f)
    print(f"Index loaded from {index_path}")
    return index, image_paths

OUTPUT_INDEX_PATH = "/content/vector.index"
faiss_index, dataset = load_faiss_index(OUTPUT_INDEX_PATH)


Index loaded from /content/vector.index


In [ ]:
import faiss
from faiss import normalize_L2
from typing import Optional, List
def add_to_index(output_path: str, bulk_embeddings: Optional[List] = None):
  print(f"output_path is {output_path}")
  faiss_index = faiss.read_index(output_path)
  last_id = faiss.vector_to_array(faiss_index.id_map).max()
  print(f"Last id is: {last_id}")
  new_start = last_id + 1
  print(f"Start of new data is: {new_start}")
  new_embeddings = bulk_embeddings[new_start:]
  print(f"Our new embeddings are: {new_embeddings}")

  with torch.no_grad():
      for i, entry in enumerate(tqdm(new_embeddings)):
        id, label, img_path = get_label_path_index(entry, i)

        embeddings = dinov2_vits14(load_image(img_path).to(device))

        embedding = embeddings[0].cpu().numpy()

        embedding = np.array(embedding).reshape(1, -1)

        #vectors need to be normalized both before adding to the index and before searching
        normalize_L2(embedding) #for the sake of using cosine similarity

        faiss_index.add_with_ids(embedding, id)

  with open(output_path + '.paths.json', 'w') as f:
      json.dump(bulk_embeddings, f)

  faiss.write_index(faiss_index, OUTPUT_INDEX_PATH)
  print(f"Index created and saved to {output_path}")

  return faiss_index

In [ ]:
print(f"Length of dataset before A: {len(dataset)}")
dataset = generate_dict_from_set(dataset, "./dataset-A/train")
print(f"Length of dataset after A: {len(dataset)}")
print(dataset)

OUTPUT_INDEX_PATH = "/content/vector.index"
faiss_index = add_to_index(output_path=OUTPUT_INDEX_PATH, bulk_embeddings=dataset)

print(f"Length of dataset before C: {len(dataset)}")
dataset = generate_dict_from_set(dataset, "./dataset-C/train")
print(f"Length of dataset after C: {len(dataset)}")
print(dataset)

OUTPUT_INDEX_PATH = "/content/vector.index"
faiss_index = add_to_index(output_path=OUTPUT_INDEX_PATH, bulk_embeddings=dataset)

Length of dataset before A: 2434
Length of dataset after A: 3241
[{'id': 0, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016830_0.jpg'}, {'id': 1, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016831_0.jpg'}, {'id': 2, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016828_0.jpg'}, {'id': 3, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016829_0.jpg'}, {'id': 4, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH017057_0.jpg'}, {'id': 5, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016826_0.jpg'}, {'id': 6, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH017054_0.jpg'}, {'id': 7, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016827_0.jpg'}, {'id': 8, 'label': 'L-MB', 'path': './dataset/train/L-MB/GH016494_0.jpg'}, {'id': 9, 'label': 'L-MB', 'path': './dataset/train/L-MB/GH013387_0.jpg'}, {'id': 10, 'label': 'L-MB', 'path': './dataset/train/L-MB/GH011803_0.jpg'}, {'id': 11, 'label': 'L-MB', 'path': './dataset/train/L-MB/GH015970_0.jpg'}, {'id': 12, 'label': 'L-MB', 'pat

  0%|          | 0/807 [00:00<?, ?it/s]

Index created and saved to /content/vector.index
Length of dataset before C: 3241
Length of dataset after C: 7285
[{'id': 0, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016830_0.jpg'}, {'id': 1, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016831_0.jpg'}, {'id': 2, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016828_0.jpg'}, {'id': 3, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016829_0.jpg'}, {'id': 4, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH017057_0.jpg'}, {'id': 5, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016826_0.jpg'}, {'id': 6, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH017054_0.jpg'}, {'id': 7, 'label': 'P-XX', 'path': './dataset/train/P-XX/GH016827_0.jpg'}, {'id': 8, 'label': 'L-MB', 'path': './dataset/train/L-MB/GH016494_0.jpg'}, {'id': 9, 'label': 'L-MB', 'path': './dataset/train/L-MB/GH013387_0.jpg'}, {'id': 10, 'label': 'L-MB', 'path': './dataset/train/L-MB/GH011803_0.jpg'}, {'id': 11, 'label': 'L-MB', 'path': './dataset/train/L-MB/G

  0%|          | 0/4044 [00:00<?, ?it/s]

Index created and saved to /content/vector.index


In [ ]:
from collections import Counter
import statistics
from faiss import normalize_L2

def majority_voting_cosine(faiss_index, embeddings, files):

  correct_guesses = []
  correct_distances = []
  incorrect_distances = []
  incorrect_guesses = []
  all_distances = []

  for i, entry in enumerate(tqdm(files)):
    id, label, img_path = get_label_path_index(entry, i)

    with torch.no_grad():
      query_vectors = dinov2_vits14(load_image(img_path).to(device))

      query_vector = query_vectors[0].cpu().numpy()

      query_vector = np.array(query_vector).reshape(1, -1)

      normalize_L2(query_vector) #have to normalize to do cosine similarity


    k = 5  # Number of nearest neighbors to retrieve
    distances, indices = faiss_index.search(query_vector, k)

    all_guesses = [] #all labels of nearest neighbors
    neighbor_distances = []

    for i, index in enumerate(indices[0]):
      #print(f"index: {index}")
      #print(f"files len: {len(files)}")
      distance = distances[0][i]

      #kind of a mess. getting the associated embeddings id/label with our neighbor index
      id = embeddings[index]['id']
      new_label = embeddings[index]['label']

      all_guesses.append(new_label)
      all_distances.append(distance)
      neighbor_distances.append(distance)
      print(f"Nearest neighbor {i+1}: {id}, {new_label} Distance {distance}")

    majority_vote = Counter(all_guesses)
    winner = sorted(all_guesses, key=lambda x: majority_vote[x], reverse=True)[0]
    if winner == label:
      print(f"most common label was {winner} which == original label {label}")
      correct_guesses.append(winner)
      correct_distances.extend(neighbor_distances)
    else:
      print(f"most common label was {winner} which != {label}")
      incorrect_guesses.append(winner)
      incorrect_distances.extend(neighbor_distances)

  print(f"Total accuracy: {len(correct_guesses)/(len(correct_guesses)+len(incorrect_guesses))}")

  print(f"Median of all distances: {statistics.median(all_distances)}")

  print(f"Median distance of incorrect guesses: {statistics.median(incorrect_distances)}")

  print(f"Median distance of correct guesses: {statistics.median(correct_distances)}")

  print(f"Lowest: {min(all_distances)} highest: {max(all_distances)}")



In [ ]:
val_dataset = generate_dict_from_set(None, "./dataset-A/val")
majority_voting_cosine(faiss_index, dataset, val_dataset)

  0%|          | 0/176 [00:00<?, ?it/s]

Nearest neighbor 1: 2529, L-MB Distance 0.9298771023750305
Nearest neighbor 2: 2462, L-MB Distance 0.9266335368156433
Nearest neighbor 3: 2538, L-MB Distance 0.9222607612609863
Nearest neighbor 4: 2524, L-MB Distance 0.9183186888694763
Nearest neighbor 5: 2535, L-MB Distance 0.9171615839004517
most common label was L-MB which == original label L-MB
Nearest neighbor 1: 2442, L-MB Distance 0.9256044626235962
Nearest neighbor 2: 2509, L-MB Distance 0.9224965572357178
Nearest neighbor 3: 2548, L-MB Distance 0.9209810495376587
Nearest neighbor 4: 2534, L-MB Distance 0.9154050350189209
Nearest neighbor 5: 2484, L-MB Distance 0.9122673869132996
most common label was L-MB which == original label L-MB
Nearest neighbor 1: 2492, L-MB Distance 0.9246180057525635
Nearest neighbor 2: 2440, L-MB Distance 0.9189257621765137
Nearest neighbor 3: 2513, L-MB Distance 0.9156588315963745
Nearest neighbor 4: 2467, L-MB Distance 0.9134962558746338
Nearest neighbor 5: 2501, L-MB Distance 0.9114207625389099
mos

In [ ]:
val_dataset = generate_dict_from_set(None, "./dataset-B/val")
majority_voting_cosine(faiss_index, dataset, val_dataset)

  0%|          | 0/550 [00:00<?, ?it/s]

Nearest neighbor 1: 1860, L-XX Distance 0.9221013188362122
Nearest neighbor 2: 0, P-XX Distance 0.9084569811820984
Nearest neighbor 3: 5, P-XX Distance 0.9076516628265381
Nearest neighbor 4: 6, P-XX Distance 0.9070166349411011
Nearest neighbor 5: 1796, L-XX Distance 0.9050424695014954
most common label was P-XX which == original label P-XX
Nearest neighbor 1: 0, P-XX Distance 0.9304468035697937
Nearest neighbor 2: 4, P-XX Distance 0.9284451603889465
Nearest neighbor 3: 3, P-XX Distance 0.9272336959838867
Nearest neighbor 4: 5, P-XX Distance 0.9272280931472778
Nearest neighbor 5: 1860, L-XX Distance 0.921693742275238
most common label was P-XX which == original label P-XX
Nearest neighbor 1: 4, P-XX Distance 0.923708975315094
Nearest neighbor 2: 0, P-XX Distance 0.9191379547119141
Nearest neighbor 3: 6, P-XX Distance 0.9094847440719604
Nearest neighbor 4: 1860, L-XX Distance 0.906001627445221
Nearest neighbor 5: 1855, L-XX Distance 0.9037492275238037
most common label was P-XX which == 

In [ ]:
val_dataset = generate_dict_from_set(None, "./dataset-C/val")
majority_voting_cosine(faiss_index, dataset, val_dataset)

  0%|          | 0/1174 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
Nearest neighbor 4: 4602, -B Distance 0.8972949981689453
Nearest neighbor 5: 4623, -B Distance 0.8954867124557495
most common label was -B which == original label -B
Nearest neighbor 1: 4599, -B Distance 0.9391987323760986
Nearest neighbor 2: 4583, -B Distance 0.9330987930297852
Nearest neighbor 3: 4564, -B Distance 0.9254971742630005
Nearest neighbor 4: 4598, -B Distance 0.9212541580200195
Nearest neighbor 5: 4643, -B Distance 0.9180120229721069
most common label was -B which == original label -B
Nearest neighbor 1: 4630, -B Distance 0.9645999073982239
Nearest neighbor 2: 4619, -B Distance 0.9572534561157227
Nearest neighbor 3: 4643, -B Distance 0.9567174911499023
Nearest neighbor 4: 4599, -B Distance 0.9512559771537781
Nearest neighbor 5: 4602, -B Distance 0.948338508605957
most common label was -B which == original label -B
Nearest neighbor 1: 4601, -B Distance 0.9662339091300964
Nearest neighbor 2: 4632, -B Distance 0.9628319144248

In [ ]:
!cp vector.index.paths.json /content/drive/MyDrive/FAISS_index/DINOv2/Dataset_ALL/
!cp vector.index /content/drive/MyDrive/FAISS_index/DINOv2/Dataset_ALL/